# Imports

In [14]:
import pandas as pd
from pathlib import Path
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

# GPU setup
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("🚀 MPS GPU!")
else:
    device = torch.device('cpu')
    print("⚠️ CPU")
print(f"Device: {device}")

🚀 MPS GPU!
Device: mps


# Data inladen 

In [15]:
yawndir = Path('/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/dataset/yawn-no-yawn')
df_yawn = []

for label_name, label_id in [('no yawn', 0), ('yawn', 1)]:
    label_path = yawndir / label_name
    img_paths = (list(label_path.rglob('*.jpg')) + 
                 list(label_path.rglob('*.png')) + 
                 list(label_path.rglob('*.jpeg')) + 
                 list(label_path.rglob('*.JPG')))
    
    print(f"{label_name}: {len(img_paths)} images")
    
    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            df_yawn.append({'Image': img, 'Label': label_id})

df_yawn = pd.DataFrame(df_yawn)
print(f"✅ Totaal: {df_yawn.shape[0]} images")
print(df_yawn['Label'].value_counts().sort_index())

no yawn: 2591 images
yawn: 2528 images
✅ Totaal: 5119 images
Label
0    2591
1    2528
Name: count, dtype: int64


# Dataset + Dataloaders

In [16]:
class SimpleYawnDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        img = self.df.iloc[idx]['Image']
        label = self.df.iloc[idx]['Label']
        
        img = cv2.resize(img, (224, 224))
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        
        if self.transform:
            img = self.transform(torch.from_numpy(img))
        return img, torch.tensor(label)

train_df, val_df = train_test_split(df_yawn, test_size=0.2, stratify=df_yawn['Label'], random_state=42)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_val = transforms.Compose([transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

train_ds = SimpleYawnDataset(train_df, transform_train)
val_ds = SimpleYawnDataset(val_df, transform_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
print("✅ Dataloaders klaar!")

✅ Dataloaders klaar!


# Model + Optimizer 

In [17]:
model_yawn = models.efficientnet_b0(weights='DEFAULT')
model_yawn.classifier[1] = nn.Linear(model_yawn.classifier[1].in_features, 2)
model_yawn = model_yawn.to(device)

optimizer = Adam(model_yawn.parameters(), lr=0.001)
criterion = CrossEntropyLoss()
print("✅ Model geladen!")

✅ Model geladen!


# Train/VAl


In [18]:
def train_epoch(model, loader, opt, crit):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt.zero_grad()
        preds = model(imgs)
        loss = crit(preds, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (preds.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss/total, correct/total

def val_epoch(model, loader, crit):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs)
            loss = crit(preds, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss/total, correct/total

print("✅ Train/val functies klaar!")

✅ Train/val functies klaar!


# Training loop


In [19]:
print("🚀 START TRAINING!")
for epoch in range(15):
    tr_loss, tr_acc = train_epoch(model_yawn, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model_yawn, val_loader, criterion)
    print(f"Epoch {epoch+1:2d}: Train {tr_acc:.3f} | Val {val_acc:.3f}")

torch.save(model_yawn.state_dict(), 'yawn_model.pth')
print("✅ YAWN MODEL OPSLAAN!")

🚀 START TRAINING!
Epoch  1: Train 0.960 | Val 0.972
Epoch  2: Train 0.981 | Val 0.986
Epoch  3: Train 0.986 | Val 0.988
Epoch  4: Train 0.989 | Val 0.990
Epoch  5: Train 0.991 | Val 0.987
Epoch  6: Train 0.994 | Val 0.984
Epoch  7: Train 0.993 | Val 0.982
Epoch  8: Train 0.991 | Val 0.987
Epoch  9: Train 0.992 | Val 0.983
Epoch 10: Train 0.995 | Val 0.983
Epoch 11: Train 0.993 | Val 0.987
Epoch 12: Train 0.998 | Val 0.984
Epoch 13: Train 0.993 | Val 0.986
Epoch 14: Train 0.992 | Val 0.973
Epoch 15: Train 0.993 | Val 0.981
✅ YAWN MODEL OPSLAAN!
